<h4> STADIOalot Return Prediction </h4>
<h4> Notebook 2 — Feature Engineering </h4>

<h4> Purpose </h4>
This notebook transforms the preprocessed dataset into features suitable for machine-learning models. Feature engineering is performed using information that would reasonably be available before the return occurs.

<h4> Input </h4>
`data/processed/returns_preprocessed.csv`

<h4> Output </h4>
`data/processed/returns_features.csv`

<h4> Feature engineering </h4>
The notebook creates:

- Order year
- Order month
- Order day
- Order day of week
- Order week of year
- Weekend indicator
- Estimated order gross value
- Discount fraction
- Estimated discount amount
- Customer age bands

Identifiers are removed from the final modelling dataset as they identify individual entities rather than representing generalisable behavioural characteristics. Such as: 
- Order_ID 
- Product_ID 
- User_ID 



In [2]:
# ============================================================
# 1. IMPORT LIBRARIES AND CONFIGURE DISPLAY
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np


# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.3f}".format)


print("=" * 70)
print("STADIOalot RETURN PREDICTION — FEATURE ENGINEERING")
print("=" * 70)
print("Libraries loaded successfully.")


# ============================================================
# 2. DEFINE PROJECT PATHS
# ============================================================

# Identify the project root
ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

ROOT = ROOT.resolve()


# Define full absolute paths
INPUT_PATH = (
    ROOT
    / "data"
    / "processed"
    / "returns_preprocessed.csv"
).resolve()


OUTPUT_PATH = (
    ROOT
    / "data"
    / "processed"
    / "returns_features.csv"
).resolve()


# Create output directory if it does not exist
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Display project paths 
paths_table = pd.DataFrame({
    "Item": [
        "Project Root",
        "Input Dataset",
        "Output Dataset"
    ],
    "Full Path": [
        str(ROOT),
        str(INPUT_PATH),
        str(OUTPUT_PATH)
    ],
    "Status": [
        "FOUND" if ROOT.exists() else "NOT FOUND",
        "FOUND" if INPUT_PATH.exists() else "NOT FOUND",
        "READY" if OUTPUT_PATH.parent.exists() else "NOT READY"
    ]
})


# Display the complete table
display(paths_table)


# Load preprocessed dataset
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Input dataset could not be found:\n{INPUT_PATH}"
    )

df = pd.read_csv(INPUT_PATH)

STADIOalot RETURN PREDICTION — FEATURE ENGINEERING
Libraries loaded successfully.


,Item,Full Path,Status
0,Project Root,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction,FOUND
1,Input Dataset,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\processed\returns_preprocessed.csv,FOUND
2,Output Dataset,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\processed\returns_features.csv,READY


In [3]:
# ============================================================
# 3. LOAD PREPROCESSED DATASET
# ============================================================

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Input dataset could not be found at:\n{INPUT_PATH}"
    )


# Order_Date was already converted to datetime
# during Notebook 1 — Data Preprocessing.
# parse_dates loads the saved date values back as datetime.
df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["Order_Date"]
)


print("Preprocessed dataset successfully loaded.")


# ============================================================
# 4. INPUT DATASET INSPECTION
# ============================================================

input_structure = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null Values": df.notna().sum().values,
    "Missing Values": df.isna().sum().values,
    "Unique Values": df.nunique(dropna=True).values
})


print("Input Structure")
display(input_structure)


# ============================================================
# 5. DATASET PREVIEW
# ============================================================

print("First five records:")
display(df.head())


Preprocessed dataset successfully loaded.
Input Structure


,Column,Data Type,Non-Null Values,Missing Values,Unique Values
0,Order_ID,str,10000,0,10000
1,Product_ID,str,10000,0,10000
2,User_ID,str,10000,0,10000
3,Order_Date,datetime64[us],10000,0,731
4,Product_Category,str,10000,0,5
5,Product_Price,float64,10000,0,9074
6,Order_Quantity,int64,10000,0,5
7,Return_Status,int64,10000,0,2
8,User_Age,int64,10000,0,53
9,User_Gender,str,10000,0,2


First five records:


,Order_ID,Product_ID,User_ID,Order_Date,Product_Category,Product_Price,Order_Quantity,Return_Status,User_Age,User_Gender,User_Location,Payment_Method,Shipping_Method,Discount_Applied
0,ORD00000000,PROD00000000,USER00000000,2023-08-05,Clothing,411.590,3,1,58,Male,City54,Debit Card,Next-Day,45.270
1,ORD00000001,PROD00000001,USER00000001,2023-10-09,Books,288.880,3,1,68,Female,City85,Credit Card,Express,47.790
2,ORD00000002,PROD00000002,USER00000002,2023-05-06,Toys,390.030,5,0,22,Female,City30,Debit Card,Next-Day,26.640
3,ORD00000003,PROD00000003,USER00000003,2024-08-29,Toys,401.090,3,0,40,Male,City95,PayPal,Next-Day,15.370
4,ORD00000004,PROD00000004,USER00000004,2023-01-16,Books,110.090,4,0,34,Female,City80,Gift Card,Standard,16.370


In [4]:
# ============================================================
# 6. VERIFY DATE COLUMNS
# ============================================================

# Order_Date was converted during Notebook 1.
# No new date conversion is performed here.

expected_date_columns = [
    "Order_Date"
]


date_check_results = []


for column in expected_date_columns:

    if column in df.columns:

        data_type = df[column].dtype

        is_datetime = (
            pd.api.types.is_datetime64_any_dtype(
                df[column]
            )
        )

        date_check_results.append({
            "Column": column,
            "Data Type": str(data_type),
            "Datetime Verified": (
                "Yes" if is_datetime else "No"
            ),
            "Status": (
                "Ready for feature extraction"
                if is_datetime
                else "CHECK — expected datetime type"
            )
        })

    else:

        date_check_results.append({
            "Column": column,
            "Data Type": "Not Found",
            "Datetime Verified": "No",
            "Status": "CHECK — column not found"
        })


# Create verification report
date_check_report = pd.DataFrame(
    date_check_results
)


print("Date Verification Report")
display(date_check_report)

Date Verification Report


,Column,Data Type,Datetime Verified,Status
0,Order_Date,datetime64[us],Yes,Ready for feature extraction


In [5]:
# ============================================================
# 7. CREATE TEMPORAL FEATURES
# ============================================================

# Date columns verified during the previous step
date_columns = [
    column
    for column in expected_date_columns
    if column in df.columns
    and pd.api.types.is_datetime64_any_dtype(df[column])
]


temporal_features_created = []


# Create temporal features from verified date columns
for column in date_columns:

    prefix = (
        column.lower()
        .replace("_date", "")
        .replace("date", "")
        .strip("_")
    )


    if prefix == "":
        prefix = "order"


    # --------------------------------------------------------
    # Year
    # --------------------------------------------------------

    year_column = f"{prefix}_year"

    df[year_column] = df[column].dt.year

    temporal_features_created.append({
        "Original Column": column,
        "Feature Created": year_column,
        "Description": "Calendar year"
    })


    # --------------------------------------------------------
    # Month
    # --------------------------------------------------------

    month_column = f"{prefix}_month"

    df[month_column] = df[column].dt.month

    temporal_features_created.append({
        "Original Column": column,
        "Feature Created": month_column,
        "Description": "Calendar month (1–12)"
    })


    # --------------------------------------------------------
    # Day
    # --------------------------------------------------------

    day_column = f"{prefix}_day"

    df[day_column] = df[column].dt.day

    temporal_features_created.append({
        "Original Column": column,
        "Feature Created": day_column,
        "Description": "Day of month (1–31)"
    })


    # --------------------------------------------------------
    # Day of Week
    # --------------------------------------------------------

    weekday_column = f"{prefix}_day_of_week"

    df[weekday_column] = df[column].dt.dayofweek

    temporal_features_created.append({
        "Original Column": column,
        "Feature Created": weekday_column,
        "Description": "Day of week (0 = Monday, 6 = Sunday)"
    })


    # --------------------------------------------------------
    # Week of Year
    # --------------------------------------------------------

    week_column = f"{prefix}_week_of_year"

    df[week_column] = (
        df[column]
        .dt.isocalendar()
        .week
        .astype(int)
    )

    temporal_features_created.append({
        "Original Column": column,
        "Feature Created": week_column,
        "Description": "ISO week of year"
    })


    # --------------------------------------------------------
    # Weekend Indicator
    # --------------------------------------------------------

    weekend_column = f"{prefix}_is_weekend"

    df[weekend_column] = (
        df[column]
        .dt.dayofweek
        .ge(5)
        .astype(int)
    )

    temporal_features_created.append({
        "Original Column": column,
        "Feature Created": weekend_column,
        "Description": "1 = Weekend, 0 = Weekday"
    })


# ============================================================
# 8. DISPLAY TEMPORAL FEATURE REPORT
# ============================================================

if temporal_features_created:

    temporal_feature_report = pd.DataFrame(
        temporal_features_created
    )

else:

    temporal_feature_report = pd.DataFrame({
        "Original Column": ["None"],
        "Feature Created": ["None"],
        "Description": [
            "No verified datetime columns available"
        ]
    })


print("Temporal Features Created")
display(temporal_feature_report)

Temporal Features Created


,Original Column,Feature Created,Description
0,Order_Date,order_year,Calendar year
1,Order_Date,order_month,Calendar month (1–12)
2,Order_Date,order_day,Day of month (1–31)
3,Order_Date,order_day_of_week,"Day of week (0 = Monday, 6 = Sunday)"
4,Order_Date,order_week_of_year,ISO week of year
5,Order_Date,order_is_weekend,"1 = Weekend, 0 = Weekday"


In [6]:
# ============================================================
# 9. REMOVE ORIGINAL DATE COLUMNS
# ============================================================

# Store the number of columns before removal
columns_before = df.shape[1]


# Remove the original date columns
df = df.drop(
    columns=date_columns,
    errors="ignore"
)


# Store the number of columns after removal
columns_after = df.shape[1]


# Create removal report
date_removal_report = pd.DataFrame({
    "Metric": [
        "Columns Before Removal",
        "Original Date Columns Removed",
        "Columns After Removal"
    ],
    "Value": [
        columns_before,
        len(date_columns),
        columns_after
    ]
})


print("Original Date Column Removal Report")
display(date_removal_report)

Original Date Column Removal Report


,Metric,Value
0,Columns Before Removal,20
1,Original Date Columns Removed,1
2,Columns After Removal,19


In [7]:
# ============================================================
# 10. CREATE COLUMN LOOKUP HELPER
# ============================================================

lower_columns = {
    column.lower(): column
    for column in df.columns
}


def find_column(possible_names):
    """
    Find the first matching column without being
    sensitive to capitalisation.
    """

    for name in possible_names:

        if name.lower() in lower_columns:
            return lower_columns[name.lower()]

    return None


print("Column lookup helper created successfully.")

Column lookup helper created successfully.


In [8]:
# ============================================================
# 11. IDENTIFY COMMERCIAL VARIABLES
# ============================================================

price_column = find_column([
    "Product_Price",
    "Price",
    "Unit_Price"
])


quantity_column = find_column([
    "Order_Quantity",
    "Quantity",
    "Qty"
])


discount_column = find_column([
    "Discount_Applied",
    "Discount",
    "Discount_Rate"
])


age_column = find_column([
    "User_Age",
    "Customer_Age",
    "Age"
])


commercial_columns = pd.DataFrame({
    "Feature Type": [
        "Price",
        "Quantity",
        "Discount",
        "Customer Age"
    ],
    "Detected Column": [
        price_column if price_column else "Not found",
        quantity_column if quantity_column else "Not found",
        discount_column if discount_column else "Not found",
        age_column if age_column else "Not found"
    ],
    "Status": [
        "FOUND" if price_column else "NOT FOUND",
        "FOUND" if quantity_column else "NOT FOUND",
        "FOUND" if discount_column else "NOT FOUND",
        "FOUND" if age_column else "NOT FOUND"
    ]
})


display(commercial_columns)

,Feature Type,Detected Column,Status
0,Price,Product_Price,FOUND
1,Quantity,Order_Quantity,FOUND
2,Discount,Discount_Applied,FOUND
3,Customer Age,User_Age,FOUND


In [9]:
# ============================================================
# 12. CREATE ORDER GROSS VALUE
# ============================================================

order_value_created = False


if price_column and quantity_column:

    price = pd.to_numeric(
        df[price_column],
        errors="coerce"
    )


    quantity = pd.to_numeric(
        df[quantity_column],
        errors="coerce"
    )


    df["order_gross_value"] = (
        price * quantity
    )


    order_value_created = True


order_value_report = pd.DataFrame({
    "Feature": [
        "order_gross_value"
    ],
    "Formula": [
        "Product Price × Order Quantity"
    ],
    "Status": [
        "CREATED"
        if order_value_created
        else "NOT CREATED — required columns unavailable"
    ]
})

print("Order Value Report")
display(order_value_report)

# ============================================================
# 13. CREATE DISCOUNT FEATURES
# ============================================================

discount_features = []


if discount_column:

    discount = pd.to_numeric(
        df[discount_column],
        errors="coerce"
    )


    # Determine whether the discount is represented
    # as a percentage or fraction.
    valid_discount = discount.dropna()


    if (
        len(valid_discount) > 0
        and valid_discount.median() > 1
    ):

        discount_fraction = (
            discount / 100
        )

        conversion_method = (
            "Percentage converted to fraction"
        )

    else:

        discount_fraction = discount

        conversion_method = (
            "Already represented as fraction"
        )


    # Keep discount values within logical range
    df["discount_fraction"] = (
        discount_fraction
        .clip(
            lower=0,
            upper=1
        )
    )


    discount_features.append({
        "Feature": "discount_fraction",
        "Description": "Discount represented as a fraction",
        "Status": "CREATED"
    })


    # Estimate discount amount
    if price_column:

        price = pd.to_numeric(
            df[price_column],
            errors="coerce"
        )


        df["discount_amount_est"] = (
            price *
            df["discount_fraction"]
        )


        discount_features.append({
            "Feature": "discount_amount_est",
            "Description": "Estimated monetary discount",
            "Status": "CREATED"
        })


else:

    conversion_method = "No discount column detected"


if discount_features:

    discount_report = pd.DataFrame(
        discount_features
    )

else:

    discount_report = pd.DataFrame({
        "Feature": ["Discount features"],
        "Description": ["No discount variable available"],
        "Status": ["NOT CREATED"]
    })

print("Discount Report")
display(discount_report)

conversion_report = pd.DataFrame({
    "Discount Processing": [
        conversion_method
    ]
})

print("Conversion Report")
display(conversion_report)

Order Value Report


,Feature,Formula,Status
0,order_gross_value,Product Price × Order Quantity,CREATED


Discount Report


,Feature,Description,Status
0,discount_fraction,Discount represented as a fraction,CREATED
1,discount_amount_est,Estimated monetary discount,CREATED


Conversion Report


,Discount Processing
0,Percentage converted to fraction


In [ ]:
# ============================================================
# 14. CREATE CUSTOMER AGE BANDS
# ============================================================

age_band_created = False


if age_column:

    age = pd.to_numeric(
        df[age_column],
        errors="coerce"
    )


    df["age_band"] = pd.cut(
        age,
        bins=[
            0,
            17,
            24,
            34,
            44,
            54,
            64,
            120
        ],
        labels=[
            "<18",
            "18-24",
            "25-34",
            "35-44",
            "45-54",
            "55-64",
            "65+"
        ],
        include_lowest=True
    )


    age_band_created = True


age_band_report = pd.DataFrame({
    "Feature": [
        "age_band"
    ],
    "Age Groups": [
        "<18, 18-24, 25-34, 35-44, 45-54, 55-64, 65+"
    ],
    "Status": [
        "CREATED"
        if age_band_created
        else "NOT CREATED — age column unavailable"
    ]
})

print("Age Band Report")
display(age_band_report)

# Age Band Distribution 

if age_band_created:

    age_distribution = (
        df["age_band"]
        .value_counts(
            dropna=False,
            sort=False
        )
        .reset_index()
    )


    age_distribution.columns = [
        "Age Band",
        "Frequency"
    ]


    age_distribution["Percentage"] = (
        age_distribution["Frequency"]
        / len(df)
        * 100
    )

    print("Age Distribution")
    display(age_distribution)

else:

    print("Age-band distribution unavailable because age_band was not created.")

Age Band Report


,Feature,Age Groups,Status
0,age_band,"<18, 18-24, 25-34, 35-44, 45-54, 55-64, 65+",CREATED


Age Distribution


,Age Band,Frequency,Percentage
0,<18,0,0.000
1,18-24,1328,13.280
2,25-34,1828,18.280
3,35-44,1878,18.780
4,45-54,1881,18.810
5,55-64,1934,19.340
6,65+,1151,11.510


In [ ]:
# ============================================================
# 15. IDENTIFY IDENTIFIER COLUMNS
# ============================================================

identifier_keywords = [
    "order_id",
    "product_id",
    "user_id",
    "customer_id",
    "transaction_id"
]


identifier_columns = []


for column in df.columns:

    column_lower = column.lower()


    if (
        column_lower in identifier_keywords
        or column_lower.endswith("_id")
    ):

        identifier_columns.append(column)


if identifier_columns:

    identifier_report = pd.DataFrame({
        "Identifier Column": identifier_columns,
        "Reason for Removal": [
            "Identifier — not a meaningful predictive feature"
            for _ in identifier_columns
        ]
    })

else:

    identifier_report = pd.DataFrame({
        "Identifier Column": ["None detected"],
        "Reason for Removal": [
            "No identifier columns detected"
        ]
    })

print("Identifier Report")
display(identifier_report)


# ============================================================
# 16. REMOVE IDENTIFIER COLUMNS
# ============================================================

columns_before = df.shape[1]


df = df.drop(
    columns=identifier_columns,
    errors="ignore"
)


columns_after = df.shape[1]


identifier_removal_summary = pd.DataFrame({
    "Metric": [
        "Columns Before Removal",
        "Identifier Columns Removed",
        "Columns After Removal"
    ],
    "Value": [
        columns_before,
        len(identifier_columns),
        columns_after
    ]
})

print("Identifier Removal Summary")
display(identifier_removal_summary)

Identifier Report


,Identifier Column,Reason for Removal
0,Order_ID,Identifier — not a meaningful predictive feature
1,Product_ID,Identifier — not a meaningful predictive feature
2,User_ID,Identifier — not a meaningful predictive feature


Identifier Removal Summary


,Metric,Value
0,Columns Before Removal,23
1,Identifier Columns Removed,3
2,Columns After Removal,20


In [14]:
# ============================================================
# 17. FINAL TARGET VALIDATION
# ============================================================

target_before = len(df)

# Convert target to numeric
df["Return_Status"] = pd.to_numeric(
    df["Return_Status"],
    errors="coerce"
)


# Remove rows with missing target
missing_target = (
    df["Return_Status"]
    .isna()
    .sum()
)


df = (
    df
    .dropna(
        subset=["Return_Status"]
    )
    .reset_index(drop=True)
)


# Convert target to integer
df["Return_Status"] = (
    df["Return_Status"]
    .astype(int)
)


target_after = len(df)


target_validation = pd.DataFrame({
    "Validation Check": [
        "Target column exists",
        "Missing target values",
        "Rows before validation",
        "Rows after validation",
        "Target values are 0 or 1"
    ],
    "Result": [
        "Return_Status" in df.columns,
        missing_target,
        target_before,
        target_after,
        df["Return_Status"].isin([0, 1]).all()
    ]
})

print("Target Validation")
display(target_validation)


Target Validation


,Validation Check,Result
0,Target column exists,True
1,Missing target values,0
2,Rows before validation,10000
3,Rows after validation,10000
4,Target values are 0 or 1,True


In [16]:
# ============================================================
# 18. FEATURE INVENTORY
# ============================================================

feature_inventory = pd.DataFrame({
    "Feature Number": range(
        1,
        len(df.columns) + 1
    ),
    "Feature": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isna().sum().values,
    "Unique Values": df.nunique(dropna=True).values
})

print("Feature Inventory")
display(feature_inventory)

# ============================================================
# 19. FEATURE-ENGINEERED MISSING-VALUE REPORT
# ============================================================

missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing Values": df.isna().sum().values
})


missing_report["Missing %"] = (
    missing_report["Missing Values"]
    / len(df)
    * 100
)


missing_report = (
    missing_report
    .sort_values(
        "Missing Values",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Missing Value Report")
display(missing_report)

Feature Inventory


,Feature Number,Feature,Data Type,Missing Values,Unique Values
0,1,Product_Category,str,0,5
1,2,Product_Price,float64,0,9074
2,3,Order_Quantity,int64,0,5
3,4,Return_Status,int64,0,2
4,5,User_Age,int64,0,53
5,6,User_Gender,str,0,2
6,7,User_Location,str,0,100
7,8,Payment_Method,str,0,4
8,9,Shipping_Method,str,0,3
9,10,Discount_Applied,float64,0,4317


Missing Value Report


,Column,Missing Values,Missing %
0,Product_Category,0,0.000
1,Product_Price,0,0.000
2,Order_Quantity,0,0.000
3,Return_Status,0,0.000
4,User_Age,0,0.000
5,User_Gender,0,0.000
6,User_Location,0,0.000
7,Payment_Method,0,0.000
8,Shipping_Method,0,0.000
9,Discount_Applied,0,0.000


In [17]:
# ============================================================
# 20. FINAL FEATURE-ENGINEERED DATASET PREVIEW
# ============================================================

display(
    df.head(10)
)

,Product_Category,Product_Price,Order_Quantity,Return_Status,User_Age,User_Gender,User_Location,Payment_Method,Shipping_Method,Discount_Applied,order_year,order_month,order_day,order_day_of_week,order_week_of_year,order_is_weekend,order_gross_value,discount_fraction,discount_amount_est,age_band
0,Clothing,411.590,3,1,58,Male,City54,Debit Card,Next-Day,45.270,2023,8,5,5,31,1,1234.770,0.453,186.327,55-64
1,Books,288.880,3,1,68,Female,City85,Credit Card,Express,47.790,2023,10,9,0,41,0,866.640,0.478,138.056,65+
2,Toys,390.030,5,0,22,Female,City30,Debit Card,Next-Day,26.640,2023,5,6,5,18,1,1950.150,0.266,103.904,18-24
3,Toys,401.090,3,0,40,Male,City95,PayPal,Next-Day,15.370,2024,8,29,3,35,0,1203.270,0.154,61.648,35-44
4,Books,110.090,4,0,34,Female,City80,Gift Card,Standard,16.370,2023,1,16,0,3,0,440.360,0.164,18.022,25-34
5,Electronics,252.120,1,1,46,Female,City47,Debit Card,Next-Day,47.610,2024,2,14,2,7,0,252.120,0.476,120.034,45-54
6,Clothing,382.890,2,1,25,Male,City50,Credit Card,Express,28.490,2023,5,29,0,22,0,765.780,0.285,109.085,25-34
7,Electronics,306.390,3,1,67,Female,City39,Credit Card,Express,38.910,2023,2,9,3,6,0,919.170,0.389,119.216,65+
8,Home,302.400,5,1,54,Male,City81,Debit Card,Standard,18.500,2024,12,25,2,52,0,1512.000,0.185,55.944,45-54
9,Clothing,294.940,3,1,44,Male,City82,Credit Card,Standard,45.210,2023,3,10,4,10,0,884.820,0.452,133.342,35-44


In [19]:
# ============================================================
# 21. FINAL DATASET SUMMARY
# ============================================================

final_summary = pd.DataFrame({
    "Metric": [
        "Final Rows",
        "Final Columns",
        "Duplicate Rows",
        "Missing Target Values",
        "Temporal Features Created",
        "Identifier Columns Removed",
        "Input Dataset",
        "Output Dataset"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df.duplicated().sum(),
        df["Return_Status"].isna().sum(),
        len(temporal_features_created),
        len(identifier_columns),
        INPUT_PATH.name,
        OUTPUT_PATH.name
    ]
})

print("Dataset Summary")
display(final_summary)

Dataset Summary


,Metric,Value
0,Final Rows,10000
1,Final Columns,20
2,Duplicate Rows,0
3,Missing Target Values,0
4,Temporal Features Created,6
5,Identifier Columns Removed,3
6,Input Dataset,returns_preprocessed.csv
7,Output Dataset,returns_features.csv


In [21]:
# ============================================================
# 22. SAVE FEATURE-ENGINEERED DATASET
# ============================================================

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)


df.to_csv(
    OUTPUT_PATH,
    index=False
)


save_report = pd.DataFrame({
    "Item": [
        "Output File",
        "Full Output Path",
        "Rows Saved",
        "Columns Saved",
        "File Exists"
    ],
    "Value": [
        OUTPUT_PATH.name,
        str(OUTPUT_PATH),
        len(df),
        len(df.columns),
        OUTPUT_PATH.exists()
    ]
})


display(save_report)
print("\nFeature Engineered dataset successfully saved.")

,Item,Value
0,Output File,returns_features.csv
1,Full Output Path,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\processed\returns_features.csv
2,Rows Saved,10000
3,Columns Saved,20
4,File Exists,True



Feature Engineered dataset successfully saved.


In [22]:
# ============================================================
# 23. FINAL VALIDATION
# ============================================================

validation_checks = pd.DataFrame({
    "Validation Check": [
        "Dataset contains rows",
        "Target column exists",
        "Target contains only 0 and 1",
        "No duplicate rows",
        "No missing target values",
        "Output file exists"
    ],
    "Result": [
        len(df) > 0,
        "Return_Status" in df.columns,
        df["Return_Status"].isin([0, 1]).all(),
        df.duplicated().sum() == 0,
        df["Return_Status"].isna().sum() == 0,
        OUTPUT_PATH.exists()
    ]
})


validation_checks["Status"] = np.where(
    validation_checks["Result"],
    "PASS",
    "CHECK"
)

print("Validation Check")
display(validation_checks)

# ============================================================
# 24. FEATURE ENGINEERING COMPLETE
# ============================================================

completion_summary = pd.DataFrame({
    "Item": [
        "Process",
        "Status",
        "Final Dataset Shape",
        "Target Variable",
        "Output File"
    ],
    "Value": [
        "Feature Engineering",
        "COMPLETE",
        str(df.shape),
        "Return_Status",
        str(OUTPUT_PATH)
    ]
})

print("Completion Summary")
display(completion_summary)

Validation Check


,Validation Check,Result,Status
0,Dataset contains rows,True,PASS
1,Target column exists,True,PASS
2,Target contains only 0 and 1,True,PASS
3,No duplicate rows,True,PASS
4,No missing target values,True,PASS
5,Output file exists,True,PASS


Completion Summary


,Item,Value
0,Process,Feature Engineering
1,Status,COMPLETE
2,Final Dataset Shape,"(10000, 20)"
3,Target Variable,Return_Status
4,Output File,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\processed\returns_features.csv
